# 03_Map_par mostrar

En este notebook usamos `folium` para mostrar la rejilla espacial que generamos en `02_agrupar_por_lat_lon.ipynb`.

- Cada punto corresponde a un cuadro de `lat | long`.
- El popup muestra la `especie` dominante y su `cantidad`.
- `folium` es ideal para mapas interactivos ligeros y para embellecer la presentación.


In [1]:
import os
import folium
from folium.plugins import MarkerCluster

try:
    import folium
    from folium.plugins import MarkerCluster
except ImportError as e:
    raise ImportError(
        "Folium no está instalado en este kernel. Ejecuta en terminal: python -m pip install folium"
    ) from e

print("folium version:", folium.__version__)


folium version: 0.20.0


In [ ]:
import os
from pyspark.sql import SparkSession

# 1. CONFIGURACIÓN DEL JAVA 17 (Vital para que no falle el spark.read)
jdk_path = os.path.expanduser('~/.jdk17')
if os.path.isdir(jdk_path):
    os.environ['JAVA_HOME'] = jdk_path
    os.environ['PATH'] = os.path.join(jdk_path, 'bin') + os.pathsep + os.environ.get('PATH', '')
    print('JAVA_HOME configurado con éxito a Java 17.')
else:
    print('Alerta: No se encontró la carpeta .jdk17 en el Home.')

# 2. SISTEMA DE RUTAS INTELIGENTE
# Buscamos el archivo en la raíz o dentro de la carpeta 'Notebook' por si acaso
base_dir = os.getcwd()
candidate_1 = os.path.join(base_dir, "datos_para_mapa_rejilla.parquet")
candidate_2 = os.path.join(base_dir, "Notebook", "datos_para_mapa_rejilla.parquet")

if os.path.isdir(candidate_1):
    parquet_path = candidate_1
elif os.path.isdir(candidate_2):
    parquet_path = candidate_2
else:
    raise FileNotFoundError(f"No encontré 'datos_para_mapa_rejilla.parquet' en:\n-{candidate_1}\n-{candidate_2}\n¡Asegúrate de haber corrido el script anterior!")

print("Leyendo datos desde:", parquet_path)

# 3. LEVANTAR SPARK
spark = SparkSession.builder \
    .appName("BiodiversidadColombiaFolium") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 4. LEER Y PASAR A PANDAS
df = spark.read.parquet(parquet_path)
print("Registros cargados:", df.count())

pandas_df = df.toPandas()
spark.stop()

# 5. MOSTRAR LAS PRIMERAS FILAS EN EL NOTEBOOK
pandas_df.head()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 06:59:33 WARN Utils: Your hostname, codespaces-e48955, resolves to a loopback address: 127.0.0.1; using 10.0.0.90 instead (on interface eth0)
26/05/29 06:59:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 06:59:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/29 06:59:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Leyendo datos desde: /workspaces/bigData-Class/Notebook/datos_para_mapa_rejilla.parquet


26/05/29 06:59:41 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

UnsupportedOperationException: getSubject is not supported

In [ ]:
import folium
from folium.plugins import MarkerCluster

if pandas_df.empty:
    raise ValueError("El DataFrame está vacío. Ejecuta primero la celda de carga de datos.")

center_lat = float(pandas_df["lat"].mean())
center_lon = float(pandas_df["long"].mean())
mapa = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="OpenStreetMap")
marker_cluster = MarkerCluster(name="Hotspots por cuadro").add_to(mapa)

def marker_color(cantidad):
    if cantidad >= 1000:
        return "darkred"
    if cantidad >= 500:
        return "red"
    if cantidad >= 100:
        return "orange"
    return "blue"

for _, row in pandas_df.iterrows():
    popup = folium.Popup(
        f"<b>Especie:</b> {row['especie']}<br>"
        f"<b>Cantidad:</b> {int(row['cantidad'])}<br>"
        f"<b>Lat:</b> {row['lat']}<br>"
        f"<b>Lon:</b> {row['long']}",
        max_width=300,
    )
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=max(4, min(16, int(row['cantidad'] / 100))),
        color=marker_color(int(row['cantidad'])),
        fill=True,
        fill_color=marker_color(int(row['cantidad'])),
        fill_opacity=0.7,
        popup=popup,
    ).add_to(marker_cluster)

title_html = '<h3 align="center">Mapa de cuadros de biodiversidad (Folium)</h3>'
mapa.get_root().html.add_child(folium.Element(title_html))

output_path = os.path.join(os.getcwd(), "03_map_cuadros_folium.html")
mapa.save(output_path)
print("Mapa guardado en:", output_path)
mapa


## ¿Qué puedes ajustar?

- Cambia el tamaño de los círculos en `radius=max(4, min(16, int(row["cantidad"] / 100)))` para que se vean mejor según tus datos.
- Cambia `round(..., 1)` en `02` por `round(..., 2)` para cuadros más pequeños.
- Si quieres etiquetas más grandes, puedes usar `folium.Marker` en lugar de `CircleMarker`.
